# NACC Cohort Builder (NACC72)
This notebook builds the **baseline CN/MCI** cohort and assigns **5-year incident outcome labels** using NACC UDS data.

**Folder conventions (as requested):**
- Inputs: `./data/`
- Code (this notebook): `./src/`
- Outputs: `./output/`

**Inputs expected in `data/`:**
- `investigator_nacc72.csv`
- `investigator_fcsf_nacc72.csv`
- `nacc_variable_dictionary_consolidated.xlsx` (or your dictionary file)

**Outputs written to `output/`:**
- `cohort.csv` (participant-level baseline features + label + optional CSF)

> You will likely only need to tweak the *etiology mapping* cell after you inspect the dictionary.


In [1]:
import os
import re
import numpy as np
import pandas as pd

DATA_DIR = os.path.join("..", "data")
OUT_DIR  = os.path.join("..", "output")
os.makedirs(OUT_DIR, exist_ok=True)

NACC_MAIN_PATH = os.path.join(DATA_DIR, "investigator_nacc72.csv")
NACC_CSF_PATH  = os.path.join(DATA_DIR, "investigator_fcsf_nacc72.csv")
DICT_PATH      = os.path.join(DATA_DIR, "nacc_variable_dictionary_consolidated.xlsx")

HORIZON_YEARS = 5

print("Main:", NACC_MAIN_PATH)
print("CSF :", NACC_CSF_PATH)
print("Dict:", DICT_PATH)


Main: ..\data\investigator_nacc72.csv
CSF : ..\data\investigator_fcsf_nacc72.csv
Dict: ..\data\nacc_variable_dictionary_consolidated.xlsx


## 1) Load the data dictionary and build a missing-code map

In [2]:
def _normalize_allowable_text(s: str) -> str:
    return s.replace(" - ", " -").replace("- ", "-")

def parse_code_labels(allowable_codes: str):
    """Parse strings like '0 = ... 1 = ... 9 = Missing -4 = Not available' into [(code,label), ...]."""
    if not isinstance(allowable_codes, str) or not allowable_codes.strip():
        return []
    txt = _normalize_allowable_text(allowable_codes)
    # Split but keep the numeric code tokens
    parts = re.split(r'(?=(?:^|\s)(-?\d+)\s*=\s*)', txt)

    out = []
    if len(parts) < 3:
        return out

    i = 1
    while i < len(parts) - 1:
        code = int(parts[i])
        rest = parts[i + 1]
        label = re.sub(r'\s+', ' ', rest).strip()
        label = label.strip(" -;")
        out.append((code, label))
        i += 2
    return out

def build_missing_code_map(dict_xlsx_path: str, variables=None):
    """Returns missing_map[var]=set(missing-like codes), label_map[var]={code: label}."""
    # Try common sheet names; fall back to first sheet
    xl = pd.ExcelFile(dict_xlsx_path)
    sheet = None
    for s in xl.sheet_names:
        if s.lower() in {"dictionary", "data dictionary", "dict"}:
            sheet = s
            break
    if sheet is None:
        sheet = xl.sheet_names[0]

    d = pd.read_excel(dict_xlsx_path, sheet_name=sheet)

    # Try to find the right column names
    # We expect something like: 'variable' and 'allowable_codes' (case-insensitive)
    cols = {c.lower(): c for c in d.columns}
    var_col = cols.get("variable") or cols.get("varname") or cols.get("name")
    allow_col = cols.get("allowable_codes") or cols.get("allowed_values") or cols.get("codes") or cols.get("values")

    if var_col is None or allow_col is None:
        raise ValueError(f"Couldn't find required columns. Found columns: {list(d.columns)}\n" 
                         "Need columns like 'variable' and 'allowable_codes'.")

    if variables is not None:
        d = d[d[var_col].isin(variables)].copy()

    missing_map = {}
    label_map = {}

    for _, r in d.iterrows():
        var = r[var_col]
        allow = r.get(allow_col, "")
        pairs = parse_code_labels(allow)

        code_to_label = {c: lbl for c, lbl in pairs}
        label_map[var] = code_to_label

        miss = set()
        for c, lbl in pairs:
            l = str(lbl).lower()
            if ("missing" in l) or ("not available" in l) or ("not applicable" in l) or ("unknown" in l):
                miss.add(c)

        missing_map[var] = miss

    return missing_map, label_map, (sheet, var_col, allow_col)

def clean_coded(series: pd.Series, missing_codes: set):
    s = pd.to_numeric(series, errors="coerce")
    if missing_codes:
        s = s.mask(s.isin(list(missing_codes)), np.nan)
    return s

missing_map, label_map, meta = build_missing_code_map(DICT_PATH)
print("Dictionary read OK:", meta)
print("Example COGSTAT codes:", list(label_map.get("COGSTAT", {}).items())[:8])
print("COGSTAT missing-like codes inferred:", missing_map.get("COGSTAT"))


Dictionary read OK: ('Dictionary', 'variable', 'allowable_codes')
Example COGSTAT codes: [(0, '0 = Clinician unable to render opinion Not applic Not applic (Form C1 or C2)'), (1, '1 = Better than normal for age TI question no. 13a (Form C2T)'), (2, '2 = Normal for age'), (3, '3 = One or two test scores abnormal TF question no. Not applic Not applic 13a (Form C2T)'), (4, '4 = Three or more scores are abnormal or lower than expected'), (9, '9 = Missing'), (-4, '4 = Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question')]
COGSTAT missing-like codes inferred: {9, -4}


In [ ]:
#pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 3.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\python312\\scripts\\pip.exe'
Consider using the `--user` option or check the permissions.



In [ ]:
#pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Load NACC main file and create visit dates

In [3]:
df = pd.read_csv(NACC_MAIN_PATH, low_memory=False)
df["NACCID"] = df["NACCID"].astype(str)

def make_visit_date(df_: pd.DataFrame) -> pd.Series:
    # Default missing month/day to mid-month/day to preserve ordering without biasing to day 1
    y = pd.to_numeric(df_["VISITYR"], errors="coerce")
    m = pd.to_numeric(df_["VISITMO"], errors="coerce").fillna(6).astype(int).clip(1, 12)
    d = pd.to_numeric(df_["VISITDAY"], errors="coerce").fillna(15).astype(int).clip(1, 28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

df["visit_date"] = make_visit_date(df)
df = df.dropna(subset=["NACCID", "visit_date"]).copy()
df = df.sort_values(["NACCID", "visit_date"]).reset_index(drop=True)

print("Rows:", len(df), "Participants:", df["NACCID"].nunique())
df[["NACCID","visit_date","COGSTAT","DEMENTED"]].head()


Rows: 207454 Participants: 55268


,NACCID,visit_date,COGSTAT,DEMENTED
0,NACC000011,2006-04-17,3,0
1,NACC000011,2007-06-18,3,0
2,NACC000011,2008-06-03,3,0
3,NACC000011,2009-08-03,2,0
4,NACC000034,2015-07-16,2,0


## 3) Clean key logic variables using dictionary missing codes

In [4]:
# Variables used for baseline logic and outcome logic
LOGIC_VARS = [
    "COGSTAT", "DEMENTED", "NORMCOG", "NACCTMCI",
    "NACCALZP", "NACCALZD",
    "VASCIF", "VASCPSIF", "VASC", "CVD",
    "NACCLBDP", "NACCLBDE", "PARK", "PD"
]

for v in LOGIC_VARS:
    if v in df.columns:
        df[v] = clean_coded(df[v], missing_map.get(v, set()))

# Special rule from your example: COGSTAT == 0 means clinician unable to render opinion (treat as unknown)
if "COGSTAT" in df.columns:
    df.loc[df["COGSTAT"] == 0, "COGSTAT"] = np.nan

df[["COGSTAT","DEMENTED","NORMCOG","NACCTMCI"]].describe(include="all")


,COGSTAT,DEMENTED,NORMCOG,NACCTMCI
count,156678.000000,207454.000000,207454.000000,207454.000000
mean,2.822745,0.288237,0.490890,6.933682
std,0.933765,0.452943,0.499918,2.331448
min,1.000000,0.000000,0.000000,1.000000
25%,2.000000,0.000000,0.000000,8.000000
50%,3.000000,0.000000,0.000000,8.000000
75%,4.000000,1.000000,1.000000,8.000000
max,4.000000,1.000000,1.000000,8.000000


## 4) Define baseline: earliest visit with CN or MCI AND no dementia

In [5]:
def first_row_per_group(df_: pd.DataFrame, group_col: str, sort_col: str) -> pd.DataFrame:
    return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()

# Baseline definition:
# - DEMENTED == 0
# - CN if NORMCOG == 1
# - MCI if NACCTMCI in {1,2,3,4}  (adjust if your dictionary says otherwise)
# If you do not have NORMCOG/NACCTMCI in your extract, you can fall back to COGSTAT rules.

if "DEMENTED" not in df.columns:
    raise ValueError("DEMENTED not found in main CSV. Please confirm the column names.")

cn_mask  = (df["NORMCOG"] == 1) if "NORMCOG" in df.columns else False
mci_mask = df["NACCTMCI"].isin([1,2,3,4]) if "NACCTMCI" in df.columns else False

# Fallback (if NORMCOG/NACCTMCI missing): treat COGSTAT 1-4 as 'not dementia' cognitive range
fallback_cogstat_mask = df["COGSTAT"].isin([1,2,3,4]) if "COGSTAT" in df.columns else False

baseline_candidates = df[(df["DEMENTED"] == 0) & (cn_mask | mci_mask | fallback_cogstat_mask)].copy()
baseline = first_row_per_group(baseline_candidates, "NACCID", "visit_date").copy()
baseline = baseline.rename(columns={"visit_date": "baseline_date"})

print("Baseline participants:", len(baseline))
baseline[["NACCID","baseline_date","COGSTAT","NORMCOG","NACCTMCI","DEMENTED"]].head()


C:\Users\branjan\AppData\Local\Temp\ipykernel_33356\2738766547.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df_.sort_values([group_col, sort_col]).groupby(group_col, as_index=False).first()


Baseline participants: 38122


,NACCID,baseline_date,COGSTAT,NORMCOG,NACCTMCI,DEMENTED
0,NACC000011,2006-04-17,3.0,0,3,0
1,NACC000034,2015-07-16,2.0,0,1,0
2,NACC000067,2012-05-23,NaN,0,3,0
3,NACC000073,2016-05-20,2.0,1,8,0
4,NACC000138,2023-07-13,4.0,0,8,0


## 5) Create 5-year window, censoring, and first dementia event

In [6]:
def years_between(a: pd.Series, b: pd.Series) -> pd.Series:
    return (b - a).dt.days / 365.25

keep_ids = set(baseline["NACCID"])
df2 = df[df["NACCID"].isin(keep_ids)].merge(
    baseline[["NACCID","baseline_date"]], on="NACCID", how="left"
).copy()

df2["t_years"] = years_between(df2["baseline_date"], df2["visit_date"])

post = df2[df2["t_years"] > 0].copy()
max_follow = post.groupby("NACCID")["t_years"].max().rename("max_follow_years")

cohort = baseline.merge(max_follow, on="NACCID", how="left")
cohort["max_follow_years"] = cohort["max_follow_years"].fillna(0)
cohort["has_5y_followup"] = cohort["max_follow_years"] >= HORIZON_YEARS

window = post[post["t_years"] <= HORIZON_YEARS].copy()
dem_events = window[window["DEMENTED"] == 1].copy()
first_dem = dem_events.sort_values(["NACCID","visit_date"]).groupby("NACCID", as_index=False).first()
first_dem = first_dem.rename(columns={"visit_date":"dementia_date"})

print("Dementia events within 5y:", len(dem_events))
print("Participants with dementia within 5y:", first_dem["NACCID"].nunique())
first_dem[["NACCID","dementia_date","DEMENTED"]].head()


Dementia events within 5y: 7525
Participants with dementia within 5y: 3882


C:\Users\branjan\AppData\Local\Temp\ipykernel_33356\2961517350.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  first_dem = dem_events.sort_values(["NACCID","visit_date"]).groupby("NACCID", as_index=False).first()


,NACCID,dementia_date,DEMENTED
0,NACC000271,2009-05-14,1
1,NACC000333,2024-07-24,1
2,NACC000626,2024-06-10,1
3,NACC001456,2024-03-08,1
4,NACC002043,2012-03-28,1


## 6) Etiology mapping (PRIMARY ONLY) — update here after you inspect dictionary

In [7]:
# IMPORTANT:
# This mapping is intentionally conservative:
# - Use primary etiology variables (e.g., NACCALZP, NACCLBDP, VASCIF/VASCPSIF) if present
# - If 0 or >1 primaries are set -> MixedOrOther (for sensitivity analyses)

def etiology_primary(row: pd.Series) -> str:
    ad_primary  = (row.get("NACCALZP") == 1)
    lbd_primary = (row.get("NACCLBDP") == 1)
    vas_primary = (row.get("VASCIF") == 1) or (row.get("VASCPSIF") == 1)

    n = int(bool(ad_primary)) + int(bool(lbd_primary)) + int(bool(vas_primary))
    if n != 1:
        return "MixedOrOther"
    if ad_primary:
        return "AD"
    if vas_primary:
        return "Vascular"
    if lbd_primary:
        return "PD_LBD"
    return "MixedOrOther"

if len(first_dem) > 0:
    first_dem["etiology_class"] = first_dem.apply(etiology_primary, axis=1)
else:
    first_dem["etiology_class"] = pd.Series(dtype=str)

first_dem["etiology_class"].value_counts(dropna=False)


etiology_class
AD              2957
MixedOrOther     546
PD_LBD           283
Vascular          96
Name: count, dtype: int64

## 7) Assign final 4-class label with censoring

In [8]:
cohort = cohort.merge(first_dem[["NACCID","dementia_date","etiology_class"]], on="NACCID", how="left")
cohort["has_dementia_5y"] = cohort["dementia_date"].notna()

def assign_label(r):
    if r["has_dementia_5y"]:
        if r["etiology_class"] in {"AD","Vascular","PD_LBD"}:
            return r["etiology_class"]
        return "MixedOrOther"
    # No dementia within window:
    if r["has_5y_followup"]:
        return "NoDementia_5y"
    return "Censored"

cohort["label"] = cohort.apply(assign_label, axis=1)

cohort["label"].value_counts(dropna=False)


label
Censored         23172
NoDementia_5y    11068
AD                2957
MixedOrOther       546
PD_LBD             283
Vascular            96
Name: count, dtype: int64

## 8) Build baseline feature table (starter set) + missingness indicators

In [ ]:
# Starter feature set — expand as you like (baseline-row values live in `baseline`)
STARTER_FEATURES = [
    "NACCID","baseline_date","label",
    "NACCAGE","SEX","RACE","HISPANIC","EDUC",
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA","DEP2YRS",
    "NACCMMSE","NACCMOCA","TRAILA","TRAILB","CDRSUM","CDRGLOB",
]

# Keep only available columns
cols = [c for c in STARTER_FEATURES if c in baseline.columns]
X = baseline[cols].copy()

# Attach label
X = X.merge(cohort[["NACCID","baseline_date","label","has_5y_followup","max_follow_years"]], 
            on=["NACCID","baseline_date"], how="left")

# Clean numeric vars using dictionary missing codes if available
NUM_VARS = ["NACCAGE","EDUC","NACCMMSE","NACCMOCA","TRAILA","TRAILB","CDRSUM","CDRGLOB","BPSYS","BPDIAS"]
for v in NUM_VARS:
    if v in X.columns:
        X[v] = clean_coded(X[v], missing_map.get(v, set()))

# Add missingness flags for key tests
for v in ["TRAILB","TRAILA","NACCMMSE","NACCMOCA","CDRSUM"]:
    if v in X.columns:
        X[f"{v}_missing"] = X[v].isna().astype(int)

X.head()


## 9) Merge CSF (nearest sample to baseline)

In [ ]:
csf = pd.read_csv(NACC_CSF_PATH, low_memory=False)
csf["NACCID"] = csf["NACCID"].astype(str)

# Convert biomarkers to numeric
for c in ["CSFABETA","CSFPTAU","CSFTTAU"]:
    if c in csf.columns:
        csf[c] = pd.to_numeric(csf[c], errors="coerce")

def make_csf_date(df_: pd.DataFrame) -> pd.Series:
    y = pd.to_numeric(df_.get("CSFLPYR"), errors="coerce")
    m = pd.to_numeric(df_.get("CSFLPMO"), errors="coerce").fillna(6).astype(int).clip(1,12)
    d = pd.to_numeric(df_.get("CSFLPDY"), errors="coerce").fillna(15).astype(int).clip(1,28)
    return pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")

csf["csf_date"] = make_csf_date(csf)

csf = csf.merge(X[["NACCID","baseline_date"]], on="NACCID", how="inner")
csf = csf.dropna(subset=["csf_date","baseline_date"]).copy()
csf["abs_days_from_baseline"] = (csf["csf_date"] - csf["baseline_date"]).abs().dt.days

csf_nearest = csf.sort_values(["NACCID","abs_days_from_baseline"]).groupby("NACCID", as_index=False).first()

merge_cols = ["NACCID","CSFABETA","CSFPTAU","CSFTTAU","abs_days_from_baseline"]
merge_cols = [c for c in merge_cols if c in csf_nearest.columns]

X2 = X.merge(csf_nearest[merge_cols], on="NACCID", how="left")
print("Rows:", len(X2), "with CSF ABETA:", X2.get("CSFABETA").notna().sum() if "CSFABETA" in X2.columns else 0)
X2.head()


## 10) Save cohort.csv

In [ ]:
OUT_PATH = os.path.join(OUT_DIR, "cohort.csv")
X2.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print(X2["label"].value_counts(dropna=False))
